### Multi-Label Enzyme-Substrate classification

In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("train.csv")
print(df.head())
df.count()

   id     BertzCT       Chi1     Chi1n      Chi1v     Chi2n      Chi2v  \
0   0  323.390782   9.879918  5.875576   5.875576  4.304757   4.304757   
1   1  273.723798   7.259037  4.441467   5.834958  3.285046   4.485235   
2   2  521.643822  10.911303  8.527859  11.050864  6.665291   9.519706   
3   3  567.431166  12.453343  7.089119  12.833709  6.478023  10.978151   
4   4  112.770735   4.414719  2.866236   2.866236  1.875634   1.875634   

      Chi3v     Chi4n  EState_VSA1  ...  SlogP_VSA3  VSA_EState9  fr_COO  \
0  2.754513  1.749203     0.000000  ...    4.794537    35.527357       0   
1  2.201375  1.289775    45.135471  ...   13.825658    44.707310       0   
2  5.824822  1.770579    15.645394  ...   17.964475    45.660120       0   
3  7.914542  3.067181    95.639554  ...   31.961948    87.509997       0   
4  1.036450  0.727664    17.980451  ...    9.589074    33.333333       2   

   fr_COO2  EC1  EC2  EC3  EC4  EC5  EC6  
0        0    1    1    0    0    0    0  
1        0  

,0
id,14838
BertzCT,14838
Chi1,14838
Chi1n,14838
Chi1v,14838
Chi2n,14838
Chi2v,14838
Chi3v,14838
Chi4n,14838
EState_VSA1,14838


### Data Preparation for Multi-Label Classification


In [3]:
X = df.drop(columns=['id', 'EC1', 'EC2', 'EC3', 'EC4', 'EC5', 'EC6'])
y = df[['EC1', 'EC2', 'EC3', 'EC4', 'EC5', 'EC6']]

In [4]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (11870, 31)
X_test shape: (2968, 31)
y_train shape: (11870, 6)
y_test shape: (2968, 6)


### Decision Tree Classifier


In [5]:
from sklearn.tree import DecisionTreeClassifier


dt_classifier = DecisionTreeClassifier(random_state=42)

dt_classifier.fit(X_train, y_train)

print("Decision Tree Classifier trained")

Decision Tree Classifier trained


### Evaluate Decision Tree Classifier

 For multi-label classification, common metrics include Jaccard score, Hamming loss, and F1-score (micro and macro averages).

In [20]:
from sklearn.metrics import jaccard_score, hamming_loss, f1_score, roc_auc_score

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    # Multi-label metrics
    jaccard = jaccard_score(y_test, y_pred, average='samples')
    hamming = hamming_loss(y_test, y_pred)
    f1_micro = f1_score(y_test, y_pred, average='micro')
    f1_macro = f1_score(y_test, y_pred, average='macro')

    print(f"Classifier Evaluation:")
    print(f"  Jaccard Score (samples): {jaccard:.4f}")
    print(f"  Hamming Loss: {hamming:.4f}")
    print(f"  F1 Score (Micro): {f1_micro:.4f}")
    print(f"  F1 Score (Macro): {f1_macro:.4f}")

    roc_aucs_per_ec = {}
    y_pred_proba_list = model.predict_proba(X_test)
    if not isinstance(y_pred_proba_list, list):
      y_pred_proba_list = [y_pred_proba_list]

    print("  ROC-AUC per EC label:")
    for i, col in enumerate(y_test.columns):
        roc_auc = roc_auc_score(y_test[col], y_pred_proba_list[i][:, 1])
        roc_aucs_per_ec[col] = roc_auc
        print(f"    {col}: {roc_auc:.4f}")

    return jaccard, hamming, f1_micro, f1_macro, roc_aucs_per_ec

res = evaluate_model(dt_classifier, X_test, y_test)
jaccard_dt, hamming_dt, f1_micro_dt, f1_macro_dt, roc_aucs_dt = res

Classifier Evaluation:
  Jaccard Score (samples): 0.4558
  Hamming Loss: 0.3297
  F1 Score (Micro): 0.5811
  F1 Score (Macro): 0.4338
  ROC-AUC per EC label:
    EC1: 0.5624
    EC2: 0.5152
    EC3: 0.5253
    EC4: 0.5508
    EC5: 0.5535
    EC6: 0.4995


### Random Forest Classifier


In [7]:
from sklearn.ensemble import RandomForestClassifier

rf_classifier = RandomForestClassifier(random_state=42)

rf_classifier.fit(X_train, y_train)

print("Random Forest Classifier trained")

Random Forest Classifier trained


### Evaluate Random Forest Classifier


In [21]:
res = evaluate_model(rf_classifier, X_test, y_test)
jaccard_rf, hamming_rf, f1_micro_rf, f1_macro_rf, roc_aucs_rf = res

Classifier Evaluation:
  Jaccard Score (samples): 0.5722
  Hamming Loss: 0.2280
  F1 Score (Micro): 0.6859
  F1 Score (Macro): 0.3921
  ROC-AUC per EC label:
    EC1: 0.6923
    EC2: 0.5574
    EC3: 0.6403
    EC4: 0.6883
    EC5: 0.6764
    EC6: 0.5555


### Gradient Boosting Machine (GBM)

In [12]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.multioutput import MultiOutputClassifier

gbm_classifier = MultiOutputClassifier(GradientBoostingClassifier(random_state=42))

gbm_classifier.fit(X_train, y_train)

print("Gradient Boosting Machine trained")

Gradient Boosting Machine trained


### Evaluate Gradient Boosting Machine

In [22]:
res = evaluate_model(gbm_classifier, X_test, y_test)
jaccard_gbm, hamming_gbm, f1_micro_gbm, f1_macro_gbm, roc_aucs_gbm = res

Classifier Evaluation:
  Jaccard Score (samples): 0.5707
  Hamming Loss: 0.2287
  F1 Score (Micro): 0.6831
  F1 Score (Macro): 0.3735
  ROC-AUC per EC label:
    EC1: 0.7016
    EC2: 0.5869
    EC3: 0.6504
    EC4: 0.7046
    EC5: 0.6978
    EC6: 0.5672


### AdaBoost classifier

In [14]:
from sklearn.ensemble import AdaBoostClassifier

ada_classifier = MultiOutputClassifier(AdaBoostClassifier(random_state=42))

ada_classifier.fit(X_train, y_train)

print("AdaBoost trained")

AdaBoost trained


### Evaluate AdaBoost Classifier

In [23]:
res = evaluate_model(ada_classifier, X_test, y_test)
jaccard_ada, hamming_ada, f1_micro_ada, f1_macro_ada, roc_aucs_ada = res

Classifier Evaluation:
  Jaccard Score (samples): 0.5715
  Hamming Loss: 0.2292
  F1 Score (Micro): 0.6866
  F1 Score (Macro): 0.3768
  ROC-AUC per EC label:
    EC1: 0.6938
    EC2: 0.5811
    EC3: 0.6486
    EC4: 0.6968
    EC5: 0.6903
    EC6: 0.5759


### Comparison of Models

In [24]:
import numpy as np

def print_results(jaccard, hamming, f1_micro, f1_macro, roc_aucs):
  print(f"Decision Tree Classifier:")
  print(f"  Jaccard Score (samples): {jaccard:.4f}")
  print(f"  Hamming Loss: {hamming:.4f}")
  print(f"  F1 Score (Micro): {f1_micro:.4f}")
  print(f"  F1 Score (Macro): {f1_macro:.4f}")
  print(f"  Average ROC-AUC: {np.mean(list(roc_aucs.values())):.4f}")
  print(f"  ROC-AUC per EC label:")
  for ec_label, auc_score in roc_aucs.items():
      print(f"{ec_label}: {auc_score:.4f}")

print(f"\nDEcision Tree Classifier:")
print_results(jaccard_dt, hamming_dt, f1_micro_dt, f1_macro_dt, roc_aucs_dt)
print(f"\nRandom Forest Classifier:")
print_results(jaccard_rf, hamming_rf, f1_micro_rf, f1_macro_rf, roc_aucs_rf)
print(f"\nGBM Classifier:")
print_results(jaccard_gbm, hamming_gbm, f1_micro_gbm, f1_macro_dt, roc_aucs_gbm)
print(f"\nAdaBoost Classifier:")
print_results(jaccard_ada, hamming_ada, f1_micro_ada, f1_macro_ada, roc_aucs_ada)


DEcision Tree Classifier:
Decision Tree Classifier:
  Jaccard Score (samples): 0.4558
  Hamming Loss: 0.3297
  F1 Score (Micro): 0.5811
  F1 Score (Macro): 0.4338
  Average ROC-AUC: 0.5344
  ROC-AUC per EC label:
EC1: 0.5624
EC2: 0.5152
EC3: 0.5253
EC4: 0.5508
EC5: 0.5535
EC6: 0.4995

Random Forest Classifier:
Decision Tree Classifier:
  Jaccard Score (samples): 0.5722
  Hamming Loss: 0.2280
  F1 Score (Micro): 0.6859
  F1 Score (Macro): 0.3921
  Average ROC-AUC: 0.6350
  ROC-AUC per EC label:
EC1: 0.6923
EC2: 0.5574
EC3: 0.6403
EC4: 0.6883
EC5: 0.6764
EC6: 0.5555

GBM Classifier:
Decision Tree Classifier:
  Jaccard Score (samples): 0.5707
  Hamming Loss: 0.2287
  F1 Score (Micro): 0.6831
  F1 Score (Macro): 0.4338
  Average ROC-AUC: 0.6514
  ROC-AUC per EC label:
EC1: 0.7016
EC2: 0.5869
EC3: 0.6504
EC4: 0.7046
EC5: 0.6978
EC6: 0.5672

AdaBoost Classifier:
Decision Tree Classifier:
  Jaccard Score (samples): 0.5715
  Hamming Loss: 0.2292
  F1 Score (Micro): 0.6866
  F1 Score (Macro): 

GBM performed best in terms of average ROC-AUC (0.65). Random Forest performed best in terms of Jaccard score, Hamming loss and F1-score (micro). F1-score micro is a global score counting all TPs, FPs, and FNs across all six classes and then applying the F1 formula. F1-score macro is calculated independently for each class and then the average of these F1 scores is taken. In our case, the micro is more relevant since enzymes can target more than one class at once. Since the competition challenge is based on ROC-AUC evaluation, I would have reported the GBM model results.